In [1]:
#cria sessão spark e app
import findspark
findspark.init()

import pyspark 
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("FT03").getOrCreate()

cores = spark._jsc.sc().getExecutorMemoryStatus().keySet().size()
print("Trabalhando com:", cores, "core(s)")
spark

Trabalhando com: 1 core(s)


In [2]:
# importa funções pyspark
from pyspark.sql.types import *
from pyspark.sql.functions import *
import pyspark.sql.functions as F
from datetime import datetime
import os
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

 ## Senadores UF

In [3]:
from re import escape
from urllib import response

import requests
import pandas as pd
import os
from datetime import datetime
from requests import status_codes
data_atual = datetime.now().strftime("%Y-%m-%d")
curl = 'https://adm.senado.gov.br/adm-dadosabertos/api/v1/senadores?formato=json'
headers = {"Accept": "application/json"}
#response  = requests.get(curl)
response  = requests.get(curl, headers=headers)
if response.status_code ==200:
        senado = response.json()
        lst_senado = senado['data'][0]
        df = pd.json_normalize(lst_senado)
       

else:
    print(response.status_code)



In [8]:
df1 = pd.DataFrame(df)

In [11]:
curl = 'https://servicodados.ibge.gov.br/api/v3/nomes/2022/nome/?tipo=nome&localidade=0'
headers = {"Accept": "application/json"}
#response  = requests.get(curl)
response  = requests.get(curl, headers=headers)
if response.status_code ==200:
        nome  = response.json()
       # lst_ = senado['data'][0]
        #df = pd.json_normalize(lst_senado)
       

else:
    print(response.status_code)


404


In [5]:
df_nome = pd.json_normalize(nome)

In [10]:
nome

{'nome': 'all',
 'tipo_nome': 'nome',
 'localidade': 0,
 'frequencia': 23,
 'percent': 0.0,
 'rank': 113027,
 'rank_letra': 11926,
 'homens': None,
 'mulheres': None,
 'idade': 29.0,
 'top_similares': [{'nome': 'aya', 'percent': 0.0002, 'frequencia': 462},
  {'nome': 'hael', 'percent': 0.0001, 'frequencia': 136},
  {'nome': 'al', 'percent': 0.0001, 'frequencia': 102},
  {'nome': 'aia', 'percent': 0, 'frequencia': 82},
  {'nome': 'haya', 'percent': 0, 'frequencia': 73},
  {'nome': 'hai', 'percent': 0, 'frequencia': 70},
  {'nome': 'ai', 'percent': 0, 'frequencia': 49},
  {'nome': 'ael', 'percent': 0, 'frequencia': 48},
  {'nome': 'ail', 'percent': 0, 'frequencia': 32},
  {'nome': 'hay', 'percent': 0, 'frequencia': 30}],
 'decadas': None,
 'upper_rank_neighbours': [{'nome': 'abde',
   'percent': 0,
   'rank': 109849,
   'frequencia': 24},
  {'nome': 'aarecida', 'percent': 0, 'rank': 106978, 'frequencia': 25},
  {'nome': 'abdan', 'percent': 0, 'rank': 104241, 'frequencia': 26},
  {'nome':

In [4]:
import requests


def obter_estados_ibge():
  """Mapeia o ID numérico do IBGE para a sigla da UF (ex: '35' -> 'SP')."""
  url = "https://ibge.gov.br"
  resposta = requests.get(url).json()
  return {str(estado["id"]): estado["sigla"] for estado in resposta}


def obter_populacao_por_uf():
  """Busca a estimativa populacional mais recente (Tabela 6579) por UF."""
  url = "https://ibge.gov.br[all]"
  resposta = requests.get(url).json()

  mapa_estados = obter_estados_ibge()
  populacao_uf = {}

  resultados = resposta[0]["resultados"][0]["series"]
  for item in resultados:
    id_uf = item["localidade"]["id"]
    sigla_uf = mapa_estados.get(id_uf)
    # Pega o valor do último período retornado
    serie = item["serie"]
    ultimo_ano = list(serie.keys())[0]
    qtd_habitantes = int(serie[ultimo_ano])

    populacao_uf[sigla_uf] = {"ano": ultimo_ano, "habitantes": qtd_habitantes}
  return populacao_uf


def obter_senadores_por_uf():
  """Lista os senadores em exercício agrupados por UF."""
  url = "https://senado.leg.br"
  headers = {"Accept": "application/json"}
  resposta = requests.get(url, headers=headers).json()

  parlamentares = resposta["ListaParlamentarEmExercicio"]["Parlamentares"]["Parlamentar"]
  senadores_uf = {}

  for p in parlamentares:
    identificacao = p["IdentificacaoParlamentar"]
    uf = identificacao["UfParlamentar"]
    nome = identificacao["NomeParlamentar"]
    partido = identificacao["SiglaPartidoParlamentar"]

    if uf not in senadores_uf:
      senadores_uf[uf] = []
    senadores_uf[uf].append(f"{nome} ({partido})")

  return senadores_uf



In [5]:

def obter_senadores_por_uf():
  """Lista os senadores em exercício agrupados por UF."""
  url = "https://adm.senado.gov.br/adm-dadosabertos/api/v1/senadores?formato=json"
  headers = {"Accept": "application/json"}
  resposta = requests.get(url, headers=headers).json()

  parlamentares = resposta["ListaParlamentarEmExercicio"]["Parlamentares"]["Parlamentar"]
  senadores_uf = {}

  for p in parlamentares:
    identificacao = p["nomeParlamentar"]
    uf = identificacao["UfParlamentar"]
    nome = identificacao["NomeParlamentar"]
    partido = identificacao["SiglaPartidoParlamentar"]

    if uf not in senadores_uf:
      senadores_uf[uf] = []
    senadores_uf[uf].append(f"{nome} ({partido})")

  return senadores_uf

In [6]:

def cruzar_dados():
  populacao = obter_populacao_por_uf()
  senadores = obter_senadores_por_uf()

  print(
      f"{'UF':<4} | {'População':>12} | {'Hab/Senador':>14} | Senadores em"
      " Exercício"
  )
  print("-" * 80)

  # Ordena da maior população para a menor
  ufs_ordenadas = sorted(
      populacao.keys(), key=lambda x: populacao[x]["habitantes"], reverse=True
  )

  for uf in ufs_ordenadas:
    hab = populacao[uf]["habitantes"]
    lista_sen = senadores.get(uf, [])
    qtd_sen = len(lista_sen)
    hab_por_senador = hab // qtd_sen if qtd_sen > 0 else 0

    nomes = ", ".join(lista_sen)
    print(f"{uf:<4} | {hab:>12,} | {hab_por_senador:>14,} | {nomes}")


if __name__ == "__main__":
  cruzar_dados()


InvalidURL: Failed to parse: https://ibge.gov.br[all]

In [ ]:
df1 = pd.DataFrame(df)

In [ ]:
df1.count()

codigoParlamentar    1
nomeParlamentar      1
titularSuplente      1
uf                   1
partido              1
endereco             1
telefones            1
fax                  1
dataNascimento       1
email                1
chefeGabinete        1
aniversario          1
mandato              1
dtype: int64

In [ ]:
df1.columns

Index(['codigoParlamentar', 'nomeParlamentar', 'titularSuplente', 'uf',
       'partido', 'endereco', 'telefones', 'fax', 'dataNascimento', 'email',
       'chefeGabinete', 'aniversario', 'mandato'],
      dtype='object')

In [ ]:
print(senado['data'])

[{'codigoParlamentar': 5672, 'nomeParlamentar': 'ALAN RICK', 'titularSuplente': 'Titular', 'uf': 'AC', 'partido': 'REPUBLICANOS', 'endereco': 'SENADO FEDERAL ANEXO 2   ALA TEOTÔNIO VILELA GABINETE 05', 'telefones': '(61)3303-6333', 'fax': '-', 'dataNascimento': '23/10/1976', 'email': 'sen.alanrick@senado.leg.br', 'chefeGabinete': 'Maria Janete Sousa Dos Santos', 'aniversario': '23/out', 'mandato': '2023 / 2031'}, {'codigoParlamentar': 5982, 'nomeParlamentar': 'ALESSANDRO VIEIRA', 'titularSuplente': 'Titular', 'uf': 'SE', 'partido': 'MDB', 'endereco': 'SENADO FEDERAL ANEXO 2   ALA AFFONSO ARINOS GABINETE 08', 'telefones': '(61)3303-9011; (61)3303-9014', 'fax': '-', 'dataNascimento': '03/04/1975', 'email': 'sen.alessandrovieira@senado.leg.br', 'chefeGabinete': 'Elaine Da Silva Gontijo', 'aniversario': '03/abr', 'mandato': '2019 / 2027'}, {'codigoParlamentar': 6358, 'nomeParlamentar': 'ANA PAULA LOBATO', 'titularSuplente': '1º Suplente', 'uf': 'MA', 'partido': 'PSB', 'endereco': 'SENADO F

In [ ]:
import pandas as pd
import requests

# 1. Faz a requisição para o endpoint em formato JSON
url = "https://legis.senado.leg.br/dadosabertos/senador/lista/atual.json"

headers = {"Accept": "application/json"}
resposta = requests.get(url, headers=headers)
dados = resposta.json()

# 2. Navega até a lista principal de interesse
lista_senadores = dados["ListaParlamentarEmExercicio"]["Parlamentares"][
    "Parlamentar"
]

# 3. Achata os dicionários internos (como IdentificacaoParlamentar e Mandato)
df = pd.json_normalize(lista_senadores)

# Exibe as colunas achatadas (ex: 'IdentificacaoParlamentar.NomeParlamentar')
#print(df[["IdentificacaoParlamentar.NomeParlamentar", "IdentificacaoParlamentar.SiglaPartidoParlamentar", "Mandato.UfParlamentar"]].head(10))
df1 = pd.DataFrame(df[["IdentificacaoParlamentar.NomeParlamentar", "IdentificacaoParlamentar.SiglaPartidoParlamentar", "Mandato.UfParlamentar"]])

In [ ]:
# importa das funções de extração de dados
from src.cria_diretorio import cria_diretorio
from src.extracao_countries import extracao_countries
from src.extracao_lauches import extracao_lauches
from src.extracao_payloads import extracao_payloads

In [ ]:
# executa funções para extração de dados das API´s
#extracao_countries()
#extracao_lauches()
extracao_payloads()

522


In [ ]:
# variavel com data atual
data_atual = datetime.now().strftime("%Y-%m-%d")

DF´s bronze - carga de dados raw

In [ ]:
# schema para carga de dados: countries
countries_schema = StructType(fields=[
                                  StructField('name',StringType(), False),
                                  StructField('official',StringType(), False),
                                  StructField('population', IntegerType(),False),
                                  StructField('gini', StringType(),False)
                                                               
                                 ])

In [ ]:
# carga de dados: countries
countries_df_bronze = spark.read.option('header',True) \
.schema(countries_schema) \
.csv(f'data/Extacao_API/data_countries_{data_atual}.csv')



In [ ]:
# schema para carga de dados: lauches
lauches_schema = StructType(fields=[
                                  StructField('id',StringType(), False),  
                                  StructField('name',StringType(), False),                                
                                  StructField('success',StringType(), False),
                                  StructField('details', StringType(),False),
                                  StructField('date_utc', StringType(),False),
                                  StructField('payloads', StringType(),False)
                                                               
                                 ])

In [ ]:
#carga de dados: lauches
lauches_df_bronze = spark.read.option('header',True) \
.schema(lauches_schema) \
.csv(f'data/Extacao_API/data_lauches_{data_atual}.csv')

In [ ]:
# schema para carga de dados: payloads
payloads_schema = StructType(fields=[
                                  StructField('id',StringType(), False),  
                                  StructField('name',StringType(), False),                                
                                  StructField('type',StringType(), False),
                                  StructField('nationalities', StringType(),False),
                                  StructField('launch', StringType(),False),
                                  StructField('customers', StringType(),False),
                                  StructField('manufacturers', StringType(),False)
                                                                                           
                                 ])

In [ ]:
#carga de dados: payloads
payloads_df_bronze = spark.read.option('header',True) \
.schema(payloads_schema) \
.csv(f'data/Extacao_API/data_payloads_{data_atual}.csv')


AnalysisException: Path does not exist: file:/c:/Users/carlo/Desktop/Udemy/Projeto_Pipeline_APIs/data/Extacao_API/data_payloads_2026-11-16.csv;

DF´s Silver - Tratamento de dados


In [ ]:
countries_df_silver = countries_df_bronze.withColumnRenamed('name','nome') \
.withColumnRenamed('official','nome_oficial') \
.withColumnRenamed('population','população')\
.drop('gini')

In [ ]:
lauches_df_silver = lauches_df_bronze.withColumn('id',lauches_df_bronze['id']) \
.withColumnRenamed('name','nome') \
.withColumnRenamed('success','sucesso')\
.withColumnRenamed('details','detalhes') \
.withColumn('data_lauche',to_date(substring(col('date_utc'),1,10))) \
.withColumn('payloads',regexp_extract('payloads',r"([a-zA-Z0-9]+)", 1)) \
.withColumn('year',year("data_lauche")) \
.drop('date_utc')


In [ ]:
payloads_df_silver = payloads_df_bronze.withColumnRenamed('name','nome') \
.withColumnRenamed('type','tipo') \
.withColumn('nacionalidade', regexp_replace((regexp_extract('nationalities', r"\[([^\]]+)\]", 1)),"\'", "") )\
.withColumn('clientes',regexp_extract('customers',r"([a-zA-Z0-9]+)", 1)) \
.drop('manufacturers') \
.drop('nationalities')\
.drop('customers')


DF´s Gold  - criação de dasets para consumo

In [ ]:
countries_df_gold =  countries_df_silver.distinct()
payloads_df_gold = payloads_df_silver.distinct()
lauches_df_gold = lauches_df_silver.distinct()

In [ ]:
path = "data/gold"

try:
    os.makedirs(path, exist_ok=True)
    print(f"Diretorio '{path}' Criado.")
except OSError as e:
    print(f"Erro na criação '{path}': {e}")

Diretorio 'data/gold' Criado.


In [ ]:
# df payloads+lauches
df_payloads_launches = payloads_df_gold.join(lauches_df_gold,payloads_df_gold.id==lauches_df_gold.payloads,'inner')\
.select(lauches_df_gold.id, \
        lauches_df_gold.data_lauche, \
        lauches_df_gold.year, \
        payloads_df_gold.nome, \
        payloads_df_gold.nacionalidade, \
        payloads_df_gold.clientes
        )

In [ ]:
#df flat para consumo
df_flat = countries_df_gold.join(df_payloads_launches,countries_df_gold.nome==df_payloads_launches.nacionalidade,'inner') \
.select(countries_df_gold.nome.alias("Pais") ,        
        countries_df_gold.nome_oficial.alias("Pais_Nome_Oficial") ,\
        #countries_df_gold.população.alias("Populacao") ,\
        df_payloads_launches.data_lauche.alias("Data_Lancamento") ,\
        df_payloads_launches.nome.alias("Satelite") ,\
        df_payloads_launches.clientes.alias("clientes"), \
        df_payloads_launches.year.alias("Ano") \
        )

dfp_flat = df_flat.toPandas()
dfp_flat.to_csv(f'data/gold/flat_{data_atual}.csv',index=False)

##### Motivo para escolha do formato .parquet, e partição por "Ano":
###### - Facilidade para querys com filtros de data
###### - Gerenciamento com baixa complexidade

In [ ]:
df_flat.coalesce(1).write.mode('overwrite').partitionBy('Ano').parquet('output/final_dataset')

In [ ]:
countries_df_gold =  countries_df_silver.distinct()
payloads_df_gold = payloads_df_silver.distinct()
lauches_df_gold = lauches_df_silver.distinct()

In [ ]:
#criação de arquivos .parquet de dataframes
countries_df_gold.coalesce(1).write.mode('overwrite').parquet('output/countries_dataset')
payloads_df_gold.coalesce(1).write.mode('overwrite').parquet('output/payloads_dataset')
lauches_df_gold.coalesce(1).write.mode('overwrite').parquet('output/lauches_dataset')

In [ ]:
#Resumo de dataset
payloads_df_gold.summary().show()

+-------+--------------------+------+-----------+--------------------+-------------+----------+
|summary|                  id|  nome|       tipo|              launch|nacionalidade|  clientes|
+-------+--------------------+------+-----------+--------------------+-------------+----------+
|  count|                 225|   225|        225|                 225|          225|       225|
|   mean|                null|  null|       null|                null|         null|      null|
| stddev|                null|  null|       null|                null|         null|      null|
|    min|5eb0e4b5b6c3bb000...|ABS-2A|Crew Dragon|5eb87cd9ffd86e000...|             |          |
|    25%|                null|  null|       null|                null|         null|      null|
|    50%|                null|  null|       null|                null|         null|      null|
|    75%|                null|  null|       null|                null|         null|      null|
|    max|63161872ffc78f3b8...|  ZUMA|  S